In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)
!pip install catboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
Q3_path = os.path.join(path, 'Q3_data.csv')
df_Q3 = pd.read_csv(Q3_path)
print(f"Dataset shape: {df_Q3.shape}")

In [ ]:
# Task 2: Write your code here:
print("\n=== First few rows ===")
print(df_Q3.head())

In [ ]:
# Task 3: Write your code here:
print("\n=== Dataset Information ===")
df_Q3.info()

In [ ]:
# Task 4: Write your code here:
print("\n=== Statistical Description ===")
print(df_Q3.describe())

In [ ]:
# Task 1: Write your code here:
print("\n=== Missing Values ===")
print(df_Q3.isnull().sum())

# Separate target column
target_col = 'target'  # Assuming target column name
if target_col not in df_Q3.columns:
    # Try to find target column (might be named differently)
    for col in df_Q3.columns:
        if 'target' in col.lower() or 'default' in col.lower() or 'label' in col.lower():
            target_col = col
            break

# Fill numerical missing values with median
for col in df_Q3.select_dtypes(include=[np.number]).columns:
    if col != target_col and df_Q3[col].isnull().sum() > 0:
        df_Q3[col].fillna(df_Q3[col].median(), inplace=True)

# Fill categorical missing values with mode
for col in df_Q3.select_dtypes(include=['object']).columns:
    if col != target_col and df_Q3[col].isnull().sum() > 0:
        df_Q3[col].fillna(df_Q3[col].mode()[0], inplace=True)

print("\nMissing values after handling:")
print(df_Q3.isnull().sum())

In [ ]:
# Task 2: Write your code here:
print(f"\n=== Duplicates ===")
print(f"Number of duplicates: {df_Q3.duplicated().sum()}")
df_Q3 = df_Q3.drop_duplicates()
print(f"Shape after removing duplicates: {df_Q3.shape}")

In [ ]:
# Task 3: Write your code here:
print("\n=== Encoding Categorical Variables ===")
categorical_cols = df_Q3.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Categorical columns: {categorical_cols}")

if len(categorical_cols) > 0:
    # Use Label Encoding for CatBoost (it handles categorical features well)
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df_Q3[col] = le.fit_transform(df_Q3[col])
        label_encoders[col] = le
    print(f"Encoded {len(categorical_cols)} categorical columns")

In [ ]:
# Task 4: Write your code here:
# Identify target column
if target_col not in df_Q3.columns:
    # Assume last column is target
    target_col = df_Q3.columns[-1]

print(f"\nTarget column: {target_col}")

X = df_Q3.drop(target_col, axis=1)
y = df_Q3[target_col]

# Store original feature names and unscaled X for later use
feature_names = X.columns.tolist()
X_unscaled = X.copy()

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n=== Feature Scaling Applied ===")
print(f"Features shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Task 5: Write your code here:
print("\n=== Target Imbalance Check ===")
print("Target distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True))

# Check if imbalanced (typically if minority class < 40%)
minority_class_ratio = y.value_counts(normalize=True).min()
if minority_class_ratio < 0.4:
    print(f"\nâ ï¸  Dataset is IMBALANCED (minority class: {minority_class_ratio*100:.2f}%)")
    print("Recommendation: Use F1 Score as evaluation metric instead of Accuracy")
else:
    print(f"\nâ Dataset is relatively BALANCED (minority class: {minority_class_ratio*100:.2f}%)")

In [ ]:
# Task 1: Write your code here:
# Task 1: Split features and target (already done above)
print("\n=== Model Training ===")

In [ ]:
# Task 2: Use StratifiedKFold (for classification with potential imbalance)
# StratifiedKFold ensures each fold has the same proportion of classes
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Determine appropriate metric based on class balance
if minority_class_ratio < 0.4:
    metric_name = "F1 Score"
    use_f1 = True
    print("Using F1 Score due to class imbalance")
else:
    metric_name = "Accuracy"
    use_f1 = False
    print("Using Accuracy (classes are balanced)")

scores = []
models = []

In [ ]:
# Task 3,4,5: Write your code here:
# Task 3, 4, 5: Train CatBoostClassifier and evaluate
for fold, (train_idx, val_idx) in enumerate(skfold.split(X_scaled, y), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoostClassifier
    model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=0
    )
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    # Calculate appropriate metric
    if use_f1:
        score = f1_score(y_val, y_pred, average='weighted')
    else:
        score = accuracy_score(y_val, y_pred)

    scores.append(score)
    models.append(model)

    print(f"Fold {fold} - {metric_name}: {score:.4f}")

# Print averaged score
print(f"\n=== Cross-Validation Results ===")
print(f"Average {metric_name} across all folds: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Train final model on full data for feature importance
final_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    random_state=42,
    verbose=0
)
final_model.fit(X_scaled, y)

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(12, 8))
top_n = min(20, len(feature_importance))
plt.barh(feature_importance['feature'][:top_n], feature_importance['importance'][:top_n])
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Feature Importance from CatBoost Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = feature_importance.iloc[0]['feature']
golden_importance = feature_importance.iloc[0]['importance']

print("\n" + "="*60)
print("GOLDEN FEATURE DISCOVERE")
print("="*60)
print(f"Feature Name: {golden_feature}")
print(f"Importance Score: {golden_importance:.4f}")
print(f"\nThis feature contributes {golden_importance:.2%} to the model's predictions!")
print("="*60)

In [ ]:
# Task Bonus: Write your code here:
# Task Bonus: Retrain with golden feature only
print("\n=== Bonus: Golden Feature Only Model ===")

# Create X with only the golden feature
X_golden = X_scaled[[golden_feature]]
print(f"Training with only: {golden_feature}")
print(f"Shape: {X_golden.shape}")

# Run StratifiedKFold with single feature
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
golden_scores = []

for fold, (train_idx, val_idx) in enumerate(skfold.split(X_golden, y), 1):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoostClassifier
    model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=0
    )
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_val)

    if use_f1:
        score = f1_score(y_val, y_pred, average='weighted')
    else:
        score = accuracy_score(y_val, y_pred)

    golden_scores.append(score)
    print(f"Fold {fold} - {metric_name}: {score:.4f}")

# Compare results
print(f"\n=== Model Comparison ===")
print(f"Full Model ({len(feature_names)} features):")
print(f"  Average {metric_name}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
print(f"\nGolden Feature Only ({golden_feature}):")
print(f"  Average {metric_name}: {np.mean(golden_scores):.4f} (+/- {np.std(golden_scores):.4f})")

performance_ratio = (np.mean(golden_scores) / np.mean(scores)) * 100
print(f"\nThe golden feature alone achieves {performance_ratio:.2f}% of the full model's performance!")

if performance_ratio > 80:
    print(f"\n Impressive! The golden feature '{golden_feature}' is powerful")
elif performance_ratio > 60:
    print(f"\n The golden feature '{golden_feature}' is quite powerful.")
else:
    print(f"\nThe golden feature is important but benefits from other features for best performance.")